<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 09. Once-for-All

## Once-for-All: Train One Network and Specialize it for Efficient Deployment

- **원 논문:** [Once-for-All: Train One Network and Specialize it for Efficient Deployment](https://openreview.net/forum?id=HylxE1HKwS)
- **저자 / 발표:** Han Cai et al. · ICLR 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 여러 width를 공유하는 supernet을 progressive shrinking으로 학습하고 latency budget에 맞는 subnet을 선택한다.

**축소하거나 생략한 부분:** 원 논문은 depth·width·kernel·resolution과 실제 여러 하드웨어를 지원한다. 실습은 width와 제공된 합성 latency table만 쓴다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Figure 1: train once, specialize | Task 1–3 | 공유 weight/subnet 선택 |
| §3.1: elastic kernel/depth/width/resolution | Task 1 | width slicing |
| §3.2: progressive shrinking | Task 2 | 큰 subnet→작은 subnet 순서 |
| §3.3: specialization | Task 3 | validation 선택 뒤 고정 subnet test 평가 |

## 미니 재현
모든 기기마다 새 모델을 훈련하지 않고 하나의 supernet weight를 잘라 여러 subnet으로 사용합니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$w^*=\arg\min_w\mathcal L_{train}(N(S,w)),\quad
s^*=\arg\max_{s\in S,\,\mathrm{lat}(s)\le b}
\mathrm{Acc}(N(s,w^*))$$

- **기호 정의:** S는 subnet 공간, w는 공유 weight, b는 배포 latency budget입니다.
- **수식의 역할:** 한 supernet을 progressive shrinking으로 학습한 뒤 장치 제약에 맞춰 전문화합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  Figure 1: train once, specialize의 흐름을 명시합니다.
- **입출력 shape:** 특징 [B, 32] → width별 hidden slice → logit [B, 4]
- **평가:** width별 accuracy, 제공 latency table의 budget 만족 여부
- **원문 대비 한계:** 원 논문은 depth·width·kernel·resolution과 실제 여러 하드웨어를 지원한다. 실습은 width와 제공된 합성
latency table만 쓴다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1: elastic kernel/depth/width/resolution
- **이 셀의 책임:** Task 1
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** width slicing. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class WidthSupernet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_weight = nn.Parameter(torch.randn(24, 32) * 0.08)
        self.hidden_bias = nn.Parameter(torch.zeros(24))
        self.out_weight = nn.Parameter(torch.randn(4, 24) * 0.08)
        self.out_bias = nn.Parameter(torch.zeros(4))

    def forward(self, x, width):
        hidden = F.relu(
            F.linear(x, self.hidden_weight[:width], self.hidden_bias[:width])
        )
        return F.linear(hidden, self.out_weight[:, :width], self.out_bias)


supernet = WidthSupernet()
assert all(supernet(features[:3], w).shape == (3, 4) for w in (8, 12, 16, 24))

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3.2: progressive shrinking
- **이 셀의 책임:** Task 2
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 큰 subnet→작은 subnet 순서. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
optimizer = torch.optim.Adam(supernet.parameters(), lr=0.03)
widths = (24, 16, 12, 8)
history = []
for phase in range(len(widths)):
    active = widths[: phase + 1]
    for _ in range(18):
        optimizer.zero_grad()
        loss = sum(
            F.cross_entropy(supernet(features[train_idx], w), labels[train_idx])
            for w in active
        ) / len(active)
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
assert history[-1] < history[0]
with torch.no_grad():
    accuracies = {
        w: (supernet(features[test_idx], w).argmax(-1) == labels[test_idx])
        .float()
        .mean()
        .item()
        for w in widths
    }
assert min(accuracies.values()) > 0.7

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 1: train once, specialize / §3.3: specialization
- **이 셀의 책임:** Task 1–3 / Task 3
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 공유 weight/subnet 선택 / validation 선택 뒤 고정 subnet test 평가. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
budget_ms = 1.3
candidates = []
for width, depth, kernel, latency in candidate_configs.tolist():
    width = int(width)
    accuracy = accuracies.get(
        width, accuracies[min(widths, key=lambda w: abs(w - width))]
    )
    candidates.append(
        {
            "width": width,
            "depth": int(depth),
            "kernel": int(kernel),
            "latency": latency,
            "accuracy": accuracy,
        }
    )
feasible = [row for row in candidates if row["latency"] <= budget_ms]
chosen = max(feasible, key=lambda row: (row["accuracy"], -row["latency"]))
assert chosen["latency"] <= budget_ms and chosen in feasible
plt.scatter([r["latency"] for r in candidates], [r["accuracy"] for r in candidates])
plt.axvline(budget_ms, color="red", linestyle="--")
plt.xlabel("synthetic latency (ms)")
plt.ylabel("accuracy")
plt.show()
print("selected subnet:", chosen)

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** Figure 1: train once, specialize / §3.1: elastic kernel/depth/width/resolution / §3.2: progressive shrinking / §3.3: specialization
- **이 셀의 책임:** Task 1–3 / Task 1 / Task 2 / Task 3
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 공유 weight/subnet 선택 / width slicing / 큰 subnet→작은 subnet 순서 / validation 선택 뒤 고정 subnet test 평가. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    widths: tuple[int, ...] = (24, 16, 12, 8)
    latency_budget_ms: float = 1.3
    learning_rate: float = 0.025
    steps_per_phase: int = 5


def prepare_batch() -> tuple[Tensor, Tensor]:
    fit_index = train_idx[:-64]
    return features[fit_index], labels[fit_index]


def width_latency_table(widths: tuple[int, ...]) -> dict[int, float]:
    """Collapse the fixture to the width-only search space implemented below."""
    rows = candidate_configs.tolist()
    table = {}
    for width in widths:
        latencies = [row[3] for row in rows if int(row[0]) == width]
        if not latencies:
            raise ValueError(f"missing latency proxy for width {width}")
        table[width] = float(min(latencies))
    return table


class OnceForAllPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        maximum = max(config.widths)
        self.hidden_weight = nn.Parameter(
            torch.randn(maximum, features.shape[1]) * 0.08
        )
        self.hidden_bias = nn.Parameter(torch.zeros(maximum))
        self.output_weight = nn.Parameter(torch.randn(4, maximum) * 0.08)
        self.output_bias = nn.Parameter(torch.zeros(4))

    def forward(self, inputs: Tensor, width: int) -> Tensor:
        hidden = F.relu(
            F.linear(
                inputs,
                self.hidden_weight[:width],
                self.hidden_bias[:width],
            )
        )
        return F.linear(hidden, self.output_weight[:, :width], self.output_bias)

    def compute_loss(
        self, inputs: Tensor, targets: Tensor, widths: tuple[int, ...]
    ) -> Tensor:
        losses = [F.cross_entropy(self(inputs, width), targets) for width in widths]
        return torch.stack(losses).mean()

    def training_step(
        self,
        inputs: Tensor,
        targets: Tensor,
        widths: tuple[int, ...],
    ) -> Tensor:
        return self.compute_loss(inputs, targets, widths)

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** Figure 1: train once, specialize / §3.1: elastic kernel/depth/width/resolution / §3.2: progressive shrinking
- **이 셀의 책임:** Task 1–3 / Task 1 / Task 2
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 공유 weight/subnet 선택 / width slicing / 큰 subnet→작은 subnet 순서. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: OnceForAllPortfolioModel) -> list[float]:
    inputs, targets = prepare_batch()
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for phase in range(len(model.config.widths)):
        active = model.config.widths[: phase + 1]
        for _ in range(model.config.steps_per_phase):
            loss = model.training_step(inputs, targets, active)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §3.3: specialization
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** validation 선택 뒤 고정 subnet test 평가. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def select_subnet_under_budget(
    model: OnceForAllPortfolioModel,
    budget_ms: float,
    validation_inputs: Tensor,
    validation_targets: Tensor,
) -> dict[str, float]:
    latency_table = width_latency_table(model.config.widths)
    candidates = []
    with torch.no_grad():
        for width, latency in latency_table.items():
            if latency <= budget_ms:
                logits = model(validation_inputs, width)
                accuracy = (logits.argmax(1) == validation_targets).float().mean()
                candidates.append(
                    {
                        "width": float(width),
                        "latency_ms": latency,
                        "validation_accuracy": float(accuracy.cpu()),
                    }
                )
    if not candidates:
        raise ValueError(f"no subnet satisfies latency budget {budget_ms}")
    return max(
        candidates,
        key=lambda item: (
            item["validation_accuracy"],
            -item["latency_ms"],
        ),
    )


def evaluate_portfolio_model(model: OnceForAllPortfolioModel) -> dict[str, float]:
    validation_index = train_idx[-64:]
    validation_inputs = features[validation_index]
    validation_targets = labels[validation_index]
    assert not torch.isin(validation_index, test_idx).any()
    budget = model.config.latency_budget_ms
    selected = select_subnet_under_budget(
        model,
        budget,
        validation_inputs,
        validation_targets,
    )
    selected_width = int(selected["width"])
    with torch.no_grad():
        test_logits = model(features[test_idx], selected_width)
        test_accuracy = (test_logits.argmax(1) == labels[test_idx]).float().mean()
    metrics = {
        "selected_test_accuracy": float(test_accuracy.cpu()),
        "fit_sample_count": float(len(train_idx) - len(validation_index)),
        "validation_sample_count": float(len(validation_index)),
        "test_sample_count": float(len(test_idx)),
    }
    metrics.update({f"selected_{key}": value for key, value in selected.items()})
    metrics["selection_within_budget"] = float(selected["latency_ms"] <= budget)
    return metrics


portfolio_model = OnceForAllPortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["selection_within_budget"] == 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 depth·width·kernel·resolution과 실제 여러 하드웨어를 지원한다. 실습은 width와 제공된 합성 latency table만 쓴다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.